# 🎨 Bild-Kolorierung mit GAN (Pix2Pix)
Dieses Notebook trainiert ein GAN-Modell, das Schwarz-Weiß-Bilder automatisch einfärbt.

**Architektur:**
- **Generator:** U-Net mit Skip-Connections
- **Discriminator:** PatchGAN
- **Farbraum:** LAB (L = Helligkeit, AB = Farbe)

## 🔧 Zelle 1 – Setup & Imports

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from skimage import color
import os

# Pakete installieren falls nötig
# !pip install scikit-image -q

print("TensorFlow Version:", tf.__version__)
print("GPU verfügbar:", tf.config.list_physical_devices('GPU'))

# Reproduzierbarkeit
tf.random.set_seed(42)
np.random.seed(42)

## 📁 Zelle 2 – Google Drive & Pfade

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Pfade anpassen ───────────────────────────────────────────
DRIVE_PATH   = "/content/drive/MyDrive/kolorierung"
IMAGE_DIR    = f"{DRIVE_PATH}/bilder"       # Hier deine Bilder ablegen
CHECKPOINT_DIR = f"{DRIVE_PATH}/checkpoints"
PREVIEW_DIR  = f"{DRIVE_PATH}/previews"
# ─────────────────────────────────────────────────────────────

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PREVIEW_DIR,    exist_ok=True)

print("✓ Drive gemountet")
print(f"✓ Bilder-Ordner: {IMAGE_DIR}")
print(f"✓ Checkpoints:  {CHECKPOINT_DIR}")

## 📁 Zelle 2b – Datensatz entzippen

In [ ]:
import zipfile, os

ZIP_PATH   = "/content/drive/MyDrive/kolorierung/train2017.zip"
IMAGE_DIR  = "/content/bilder"

if not os.path.exists(IMAGE_DIR):
    print("📦 Entzippe Datensatz...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(IMAGE_DIR)
    print(f"✓ {len(os.listdir(IMAGE_DIR))} Bilder entpackt")
else:
    print(f"✓ Bilder bereits entpackt ({len(os.listdir(IMAGE_DIR))} Stück)")

## ⚙️ Zelle 3 – Hyperparameter

In [ ]:
IMG_SIZE    = 256       # Bildgröße (256x256)
BATCH_SIZE  = 16        # Batch-Größe (bei wenig RAM auf 8 reduzieren)
EPOCHS      = 100       # Anzahl Trainings-Epochen
LAMBDA      = 100       # Gewichtung L1-Loss vs. GAN-Loss
LR          = 2e-4      # Lernrate
BETA_1      = 0.5       # Adam Beta1
SAVE_EVERY  = 5         # Checkpoint alle N Epochen

print("Hyperparameter gesetzt ✓")

## 🔄 Zelle 4 – Datenpipeline

In [ ]:
def load_and_preprocess(image_path):
    """Lädt ein Bild und konvertiert es in LAB-Farbraum."""
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0

    # Data Augmentation
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)

    img_np = img.numpy()
    lab    = color.rgb2lab(img_np).astype(np.float32)

    L  = lab[:,:,0:1] / 50.0 - 1.0   # Normalisiert auf [-1, 1]
    AB = lab[:,:,1:]  / 128.0          # Normalisiert auf [-1, 1]
    return L, AB


def make_dataset(image_dir, batch_size=BATCH_SIZE):
    """Erstellt ein tf.data.Dataset aus einem Bildordner."""
    paths = tf.data.Dataset.list_files(image_dir + "/*.jpg", shuffle=True)

    def tf_preprocess(path):
        L, AB = tf.py_function(load_and_preprocess, [path], [tf.float32, tf.float32])
        L.set_shape([IMG_SIZE, IMG_SIZE, 1])
        AB.set_shape([IMG_SIZE, IMG_SIZE, 2])
        return L, AB

    return (paths
        .map(tf_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE))


dataset = make_dataset(IMAGE_DIR)
print(f"✓ Dataset erstellt")

# Anzahl Bilder zählen
num_images = len(tf.io.gfile.glob(IMAGE_DIR + "/*.jpg"))
print(f"✓ {num_images} Bilder gefunden")
print(f"✓ {num_images // BATCH_SIZE} Steps pro Epoche")

## 🏗️ Zelle 5 – Generator (U-Net)

In [ ]:
def conv_block(x, filters, size=4, strides=2, use_bn=True):
    """Encoder-Block: Conv → BN → LeakyReLU"""
    x = tf.keras.layers.Conv2D(
        filters, size, strides=strides,
        padding='same', use_bias=False
    )(x)
    if use_bn:
        x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    return x


def deconv_block(x, skip, filters, dropout=False):
    """Decoder-Block: ConvTranspose → BN → (Dropout) → ReLU → Concat"""
    x = tf.keras.layers.Conv2DTranspose(
        filters, 4, strides=2,
        padding='same', use_bias=False
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    if dropout:
        x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Concatenate()([x, skip])
    return x


def build_generator():
    """U-Net Generator: L-Kanal → AB-Kanäle"""
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="L_input")

    # ── Encoder ──────────────────────────────
    e1 = conv_block(inputs, 64,  use_bn=False)  # 128x128
    e2 = conv_block(e1,     128)                 # 64x64
    e3 = conv_block(e2,     256)                 # 32x32
    e4 = conv_block(e3,     512)                 # 16x16
    e5 = conv_block(e4,     512)                 # 8x8
    e6 = conv_block(e5,     512)                 # 4x4

    # ── Bottleneck ───────────────────────────
    b  = conv_block(e6,     512)                 # 2x2

    # ── Decoder mit Skip-Connections ─────────
    d1 = deconv_block(b,  e6, 512, dropout=True)  # 4x4
    d2 = deconv_block(d1, e5, 512, dropout=True)  # 8x8
    d3 = deconv_block(d2, e4, 512, dropout=True)  # 16x16
    d4 = deconv_block(d3, e3, 256)                # 32x32
    d5 = deconv_block(d4, e2, 128)                # 64x64
    d6 = deconv_block(d5, e1, 64)                 # 128x128

    # ── Output ───────────────────────────────
    outputs = tf.keras.layers.Conv2DTranspose(
        2, 4, strides=2, padding='same', activation='tanh', name="AB_output"
    )(d6)  # 256x256x2

    return tf.keras.Model(inputs, outputs, name="Generator")


generator = build_generator()
print("Generator:")
generator.summary()

## 🏗️ Zelle 6 – Discriminator (PatchGAN)

In [ ]:
def build_discriminator():
    """
    PatchGAN Discriminator:
    Bewertet 70x70 Bildausschnitte statt das gesamte Bild.
    Input: L-Kanal + AB-Kanäle (real oder fake)
    Output: Patch-Map (echt=1 / falsch=0)
    """
    L  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="L_input")
    AB = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 2), name="AB_input")

    # L und AB zusammenführen → 3 Kanäle
    x = tf.keras.layers.Concatenate()([L, AB])

    # ── Schichten ────────────────────────────
    x = conv_block(x, 64,  use_bn=False)  # 128x128
    x = conv_block(x, 128)                # 64x64
    x = conv_block(x, 256)                # 32x32

    # Stride=1 für die letzten Layer (PatchGAN)
    x = tf.keras.layers.Conv2D(
        512, 4, strides=1, padding='same', use_bias=False
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)

    # ── Output: Patch-Map ────────────────────
    x = tf.keras.layers.Conv2D(1, 4, strides=1, padding='same', name="patch_output")(x)

    return tf.keras.Model([L, AB], x, name="Discriminator")


discriminator = build_discriminator()
print("Discriminator:")
discriminator.summary()

## 📉 Zelle 7 – Loss-Funktionen & Optimizers

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    """Discriminator soll Echtes als 1 und Falsches als 0 erkennen."""
    real_loss = bce(tf.ones_like(real_output),  real_output)
    fake_loss = bce(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss


def generator_loss(fake_output, pred_AB, real_AB):
    """
    Generator-Loss = GAN-Loss + Lambda * L1-Loss
    - GAN-Loss:  Discriminator überlisten (Fake als Echt ausgeben)
    - L1-Loss:   Pixelgenaue Übereinstimmung mit Original
    """
    gan_loss = bce(tf.ones_like(fake_output), fake_output)
    l1_loss  = tf.reduce_mean(tf.abs(real_AB - pred_AB))
    total    = gan_loss + LAMBDA * l1_loss
    return total, gan_loss, l1_loss


# Optimizers
gen_optimizer  = tf.keras.optimizers.Adam(LR, beta_1=BETA_1)
disc_optimizer = tf.keras.optimizers.Adam(LR, beta_1=BETA_1)

print("✓ Loss-Funktionen und Optimizers bereit")

## 💾 Zelle 8 – Checkpoint laden (optional)

In [ ]:
# ── Checkpoint laden (falls vorhanden) ──────────────────────────
# Epoche anpassen die geladen werden soll, z.B. 050
LOAD_EPOCH = None   # z.B. 50 – oder None für Neustart
# ────────────────────────────────────────────────────────────────

START_EPOCH = 0

if LOAD_EPOCH is not None:
    gen_path  = f"{CHECKPOINT_DIR}/gen_epoch_{LOAD_EPOCH:03d}.keras"
    disc_path = f"{CHECKPOINT_DIR}/disc_epoch_{LOAD_EPOCH:03d}.keras"

    if os.path.exists(gen_path) and os.path.exists(disc_path):
        generator     = tf.keras.models.load_model(gen_path)
        discriminator = tf.keras.models.load_model(disc_path)
        START_EPOCH   = LOAD_EPOCH
        print(f"✓ Checkpoint Epoche {LOAD_EPOCH} geladen – Training wird fortgesetzt")
    else:
        print(f"✗ Checkpoint nicht gefunden → starte von Epoche 0")
else:
    print("✓ Kein Checkpoint – Training startet von Epoche 0")

## 🚀 Zelle 9 – Trainingsschritt

In [ ]:
@tf.function
def train_step(L, real_AB):
    """Ein Trainingsschritt für Generator und Discriminator."""
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:

        # 1. Generator erzeugt AB-Kanäle aus L
        fake_AB = generator(L, training=True)

        # 2. Discriminator bewertet echte und falsche AB-Kanäle
        real_output = discriminator([L, real_AB], training=True)
        fake_output = discriminator([L, fake_AB], training=True)

        # 3. Losses berechnen
        disc_loss                  = discriminator_loss(real_output, fake_output)
        gen_total, gan_loss, l1_loss = generator_loss(fake_output, fake_AB, real_AB)

    # 4. Gradienten berechnen und anwenden
    gen_grads  = gen_tape.gradient(gen_total,  generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    gen_optimizer.apply_gradients(zip(gen_grads,  generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return gen_total, disc_loss, gan_loss, l1_loss


print("✓ Trainingsschritt kompiliert")

## 🖼️ Zelle 10 – Visualisierung

In [ ]:
def visualize(epoch, n=4):
    """Zeigt S/W-Eingabe, GAN-Kolorierung und Original nebeneinander."""
    L_batch, AB_batch = next(iter(dataset.take(1)))
    pred_AB = generator(L_batch[:n], training=False)

    fig, axes = plt.subplots(n, 3, figsize=(12, n * 4))
    fig.suptitle(f"Epoche {epoch}", fontsize=16, fontweight='bold')

    for i in range(n):
        L_np      = (L_batch[i].numpy()  + 1.0) * 50.0
        predAB_np =  pred_AB[i].numpy()  * 128.0
        realAB_np =  AB_batch[i].numpy() * 128.0

        pred_rgb = np.clip(color.lab2rgb(np.concatenate([L_np, predAB_np], axis=-1)), 0, 1)
        real_rgb = np.clip(color.lab2rgb(np.concatenate([L_np, realAB_np], axis=-1)), 0, 1)

        axes[i, 0].imshow(L_np[:,:,0], cmap='gray')
        axes[i, 0].set_title("Eingabe (S/W)",    fontsize=11)
        axes[i, 1].imshow(pred_rgb)
        axes[i, 1].set_title("GAN Koloriert",    fontsize=11)
        axes[i, 2].imshow(real_rgb)
        axes[i, 2].set_title("Original",          fontsize=11)

        for ax in axes[i]:
            ax.axis('off')

    plt.tight_layout()
    save_path = f"{PREVIEW_DIR}/preview_epoch_{epoch:03d}.png"
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"  📸 Vorschau gespeichert: {save_path}")


print("✓ Visualisierungsfunktion bereit")

## 🔁 Zelle 11 – Training Loop

In [ ]:
# Loss-History für spätere Auswertung
history = {
    'gen_loss':  [],
    'disc_loss': [],
    'gan_loss':  [],
    'l1_loss':   []
}

print(f"🚀 Training startet ab Epoche {START_EPOCH + 1}\n")

for epoch in range(START_EPOCH, EPOCHS):
    print(f"── Epoche {epoch + 1:3d}/{EPOCHS} ────────────────────────")

    epoch_gen,  epoch_disc = [], []
    epoch_gan,  epoch_l1   = [], []

    for step, (L, AB) in enumerate(dataset):
        g_loss, d_loss, gan_l, l1_l = train_step(L, AB)

        epoch_gen.append(float(g_loss))
        epoch_disc.append(float(d_loss))
        epoch_gan.append(float(gan_l))
        epoch_l1.append(float(l1_l))

        if step % 50 == 0:
            print(f"  Step {step:4d} │ "
                  f"G: {g_loss:.4f} (GAN: {gan_l:.4f}, L1: {l1_l:.4f}) │ "
                  f"D: {d_loss:.4f}")

    # Epochen-Durchschnitt speichern
    history['gen_loss'].append(np.mean(epoch_gen))
    history['disc_loss'].append(np.mean(epoch_disc))
    history['gan_loss'].append(np.mean(epoch_gan))
    history['l1_loss'].append(np.mean(epoch_l1))

    print(f"  ▶ Ø G-Loss: {history['gen_loss'][-1]:.4f} │ "
          f"Ø D-Loss: {history['disc_loss'][-1]:.4f}")

    # Checkpoint speichern
    if (epoch + 1) % SAVE_EVERY == 0:
        generator.save(f"{CHECKPOINT_DIR}/gen_epoch_{epoch+1:03d}.keras")
        discriminator.save(f"{CHECKPOINT_DIR}/disc_epoch_{epoch+1:03d}.keras")
        print(f"  💾 Checkpoint gespeichert (Epoche {epoch+1})")

    # Vorschau anzeigen
    if (epoch + 1) % 5 == 0:
        visualize(epoch + 1)

print("\n✅ Training abgeschlossen!")

## 📊 Zelle 12 – Loss-Kurven

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training Loss", fontsize=14, fontweight='bold')

axes[0].plot(history['gen_loss'],  label='Generator (gesamt)', color='blue')
axes[0].plot(history['disc_loss'], label='Discriminator',      color='red')
axes[0].set_title("GAN Loss")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['l1_loss'],  label='L1 Loss',  color='green')
axes[1].plot(history['gan_loss'], label='GAN Loss', color='orange')
axes[1].set_title("Generator Loss Aufschlüsselung")
axes[1].set_xlabel("Epoche")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DRIVE_PATH}/loss_curves.png", dpi=100)
plt.show()

## 🧪 Zelle 13 – Einzelbild testen

In [ ]:
def colorize_image(image_path):
    """Färbt ein einzelnes S/W-Bild ein."""
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0

    img_np = img.numpy()
    lab    = color.rgb2lab(img_np).astype(np.float32)
    L      = (lab[:,:,0:1] / 50.0 - 1.0)[np.newaxis, ...]  # Batch-Dimension

    # Vorhersage
    pred_AB = generator(L, training=False)[0].numpy() * 128.0
    L_np    = (L[0] + 1.0) * 50.0

    result_rgb = np.clip(color.lab2rgb(np.concatenate([L_np, pred_AB], axis=-1)), 0, 1)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(L_np[:,:,0], cmap='gray'); axes[0].set_title("Eingabe (S/W)")
    axes[1].imshow(result_rgb);               axes[1].set_title("GAN Koloriert")
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()
    return result_rgb


# Beispiel:
# result = colorize_image("/content/drive/MyDrive/kolorierung/test.jpg")